# NLLB-200 Backtranslation per BioMQM (Zero-Setup & Safe)

Questo notebook è progettato per essere eseguito con zero configurazione manuale:
1. **Auto-Clone**: Scarica automaticamente la repository con i dati BioMQM.
2. **Salvataggio Incrementale**: Ogni frase viene salvata subito su disco per prevenire perdite in caso di disconnessione.
3. **Merging Finale**: Ricompone il file originale completo con le nuove traduzioni.

### 1. Setup Ambiente e Repository
Esegui questa cella per scaricare i dati e caricare il modello.

In [ ]:
# @title Setup Automonatico
import os

# 1. Clone della repository se non presente
if not os.path.exists('AskQE_DNLP_2025-2026'):
    print("Cloning repository...")
    !git clone https://github.com/laurabon/AskQE_DNLP_2025-2026.git
else:
    print("Repository già presente.")

# 2. Installazione dipendenze
!pip install -q transformers accelerate torch

import json
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from google.colab import files

MODEL_NAME = "facebook/nllb-200-distilled-600M"
LANG_MAP = {
    "de": "deu_Latn",
    "es": "spa_Latn",
    "fr": "fra_Latn",
    "ru": "rus_Cyrl",
    "zh-CN": "zho_Hans",
    "en": "eng_Latn"
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print(f"Loading model {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

def run_backtranslate(lang_key):
    if lang_key not in data_by_lang:
        print(f"ERRORE: Dati per '{lang_key}' non trovati.")
        return
    
    data_list = data_by_lang[lang_key]
    lang_flores = LANG_MAP[lang_key]
    target_lang = LANG_MAP["en"]
    output_filename = f"dev_with_backtranslation_nllb_{lang_key}.jsonl"
    
    # Impostazione della lingua sorgente nel tokenizer
    tokenizer.src_lang = lang_flores
    # Recupero sicuro dell'ID della lingua target
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(target_lang)
    
    processed_count = 0
    if os.path.exists(output_filename):
        with open(output_filename, 'r') as tmp: processed_count = len(tmp.readlines())
        print(f"Riprendo dalla riga {processed_count}...")
    
    total = len(data_list)
    print(f"Avvio traduzione {lang_key.upper()}: {total} frasi...")
    
    with open(output_filename, 'a', encoding='utf-8') as f:
        for i in range(processed_count, total):
            item = data_list[i]
            text = item.get("tgt")
            
            if not text:
                item["bt_tgt"] = ""
            else:
                try:
                    inputs = tokenizer(text, return_tensors="pt").to(device)
                    translated_tokens = model.generate(
                        **inputs, 
                        forced_bos_token_id=forced_bos_token_id, 
                        max_length=128
                    )
                    item["bt_tgt"] = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
                except Exception as e:
                    print(f"\nErrore riga {i}: {e}")
                    item["bt_tgt"] = ""
            
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
            f.flush()
            
            if i % 25 == 0:
                print(f"Progress {lang_key}: {i}/{total}", end='\r')
    
    print(f"\nCOMPLETATO: {output_filename} salvato.")
    files.download(output_filename)

### 2. Caricamento Dataset BioMQM
Il file verrà caricato automaticamente dalla cartella clonato.

In [ ]:
input_file = "/content/AskQE_DNLP_2025-2026/biomqm/dev_with_backtranslation.jsonl"

if not os.path.exists(input_file):
    print("File non trovato nel percorso della repo. Caricalo manualmente se necessario:")
    uploaded = files.upload()
    if uploaded:
        input_file = list(uploaded.keys())[0]
    else:
        raise FileNotFoundError("File di input non disponibile.")

data_by_lang = {}
original_order = []

with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line.strip())
        l = item.get("lang_tgt")
        original_order.append((l, item.get("src"), item.get("tgt")))
        if l in LANG_MAP:
            if l not in data_by_lang:
                data_by_lang[l] = []
            data_by_lang[l].append(item)

print("Righe caricate:", len(original_order))
print("Lingue rilevate:", list(data_by_lang.keys()))

### 3. Esecuzione per Lingua

In [ ]:
run_backtranslate('de')

In [ ]:
run_backtranslate('es')

In [ ]:
run_backtranslate('fr')

In [ ]:
run_backtranslate('ru')

In [ ]:
run_backtranslate('zh-CN')

### 4. Merging Finale

In [ ]:
final_output = "dev_with_backtranslation_nllb.jsonl"

bt_data_map = {}
for lang in LANG_MAP.keys():
    fname = f"dev_with_backtranslation_nllb_{lang}.jsonl"
    if os.path.exists(fname):
        with open(fname, 'r', encoding='utf-8') as f:
            for line in f:
                d = json.loads(line)
                # Usiamo una chiave unica per il match
                key = (d.get('lang_tgt'), d.get('src'), d.get('tgt'))
                bt_data_map[key] = d

with open(input_file, 'r', encoding='utf-8') as f_in, \
     open(final_output, 'w', encoding='utf-8') as f_out:
    for line in f_in:
        original_item = json.loads(line.strip())
        key = (original_item.get('lang_tgt'), original_item.get('src'), original_item.get('tgt'))
        
        if key in bt_data_map:
            f_out.write(json.dumps(bt_data_map[key], ensure_ascii=False) + '\n')
        else:
            original_item["bt_tgt"] = ""
            f_out.write(json.dumps(original_item, ensure_ascii=False) + '\n')

print(f"File finale salvato: {final_output}")
files.download(final_output)